# 4 Boosting Gazizov

Финальный product-oriented тюнинг CatBoost для участника 1. В этом ноутбуке сравниваем только три стратегии, чтобы получить честный и компактный выбор кандидатов под разные продуктовые цели.

## Стратегии

Используем только:
- train_quantile_04
- train_engagement_quantile_035
- train_capped_target

Это закрывает три ключевых режима: осторожный quantile 0.40, более консервативный quantile 0.35 и capped MAE подход.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd

repo_root = Path.cwd()
if repo_root.name != 'ml_in_gamedev_project':
    for p in [repo_root, *repo_root.parents]:
        if p.name == 'ml_in_gamedev_project':
            repo_root = p
            break
sys.path.append(str(repo_root))

from catboost import CatBoostRegressor, CatBoostClassifier
from preprocessing.preprocessing import load_data, prepare_for_targets, regression_metrics, TargetTransform

pd.set_option('display.max_columns', 300)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)


## Данные и два таргета

Работаем с двумя нашими зафиксированными таргетами

In [2]:
target_cols = [
    'target_next_session_length_sec',
    'future_sessions_mean_playtime_7d',
]

df = load_data()
packs = prepare_for_targets(df, target_cols=target_cols, max_rows=30000)

pd.DataFrame([
    {
        'target': t,
        'train_rows': len(p.x_train),
        'val_rows': len(p.x_val),
        'test_rows': len(p.x_test),
        'num_cols': len(p.num_cols),
        'cat_cols': len(p.cat_cols),
    }
    for t, p in packs.items()
])

,target,train_rows,val_rows,test_rows,num_cols,cat_cols
0,target_next_session_length_sec,21000,4500,4500,59,14
1,future_sessions_mean_playtime_7d,21000,4500,4500,59,14


## Метрики

Используем базовые метрики из preprocessing и дополнительные WMAPE + TailWeightedMAE

In [3]:
def wmape(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    den = np.abs(yt).sum()
    if den <= 0:
        return np.nan
    return float(np.abs(yt - yp).sum() / den)

def tail_weighted_mae(y_true, y_pred, split_q=0.9, tail_w=2.0):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    e = np.abs(yt - yp)
    thr = np.quantile(yt, split_q)
    w = np.where(yt >= thr, tail_w, 1.0)
    return float(np.sum(w * e) / np.sum(w))

def metric_pack(y_true, y_pred):
    m = regression_metrics(y_true, y_pred)
    m['wmape'] = wmape(y_true, y_pred)
    m['tail_weighted_mae_q90_w2'] = tail_weighted_mae(y_true, y_pred, split_q=0.9, tail_w=2.0)
    return m

In [ ]:
mode_defs = [
    {'mode': 'train_quantile_04', 'kind': 'quantile_loss', 'alpha': 0.40},
    {'mode': 'train_engagement_quantile_035', 'kind': 'quantile_loss', 'alpha': 0.35},
    {'mode': 'train_capped_target', 'kind': 'transform', 'transform': 'p995'},
]

medium_grid = {
    'depth': [5, 6, 7],
    'learning_rate': [0.03, 0.05, 0.07],
    'l2_leaf_reg': [3, 5, 8],
    'iterations': [500, 800, 1100],
    'bagging_temperature': [0.0, 0.5, 1.0],
    'subsample': [0.7, 0.85, 1.0],
    'random_strength': [0.5, 1.0, 2.0],
    'border_count': [64, 128, 254],
    'od_wait': [80, 120],
}

base_params = dict(
    od_type='Iter',
    eval_metric='MAE',
    random_seed=42,
    verbose=False,
    thread_count=-1,
)

grid_rows = []
for d in medium_grid['depth']:
    for lr in medium_grid['learning_rate']:
        for l2 in medium_grid['l2_leaf_reg']:
            for it in medium_grid['iterations']:
                for bt in medium_grid['bagging_temperature']:
                    for ss in medium_grid['subsample']:
                        for rs in medium_grid['random_strength']:
                            for bc in medium_grid['border_count']:
                                for ow in medium_grid['od_wait']:
                                    grid_rows.append({
                                        'depth': d,
                                        'learning_rate': lr,
                                        'l2_leaf_reg': l2,
                                        'iterations': it,
                                        'bagging_temperature': bt,
                                        'subsample': ss,
                                        'random_strength': rs,
                                        'border_count': bc,
                                        'od_wait': ow,
                                    })


param_grid = pd.DataFrame(grid_rows).drop_duplicates().sample(n=min(72, len(grid_rows)), random_state=42).to_dict('records')
len(param_grid)

72

In [5]:
def fit_predict_mode(p, mdef, hp, shrink_rate=0.0):
    xtr, xva, xte = p.x_train, p.x_val, p.x_test
    ytr = p.y_train
    cat_cols = p.cat_cols

    if mdef['kind'] == 'transform':
        tfm = TargetTransform(mode=mdef['transform']).fit(ytr)
        yfit = tfm.transform(ytr)
        reg = CatBoostRegressor(loss_function='MAE', **base_params, **hp)
        t0 = time.time()
        reg.fit(xtr, yfit, cat_features=cat_cols)
        fit_sec = time.time() - t0
        pr_tr = tfm.inverse(reg.predict(xtr))
        pr_va = tfm.inverse(reg.predict(xva))
        pr_te = tfm.inverse(reg.predict(xte))
        return pr_tr, pr_va, pr_te, fit_sec

    alpha = mdef['alpha']
    reg = CatBoostRegressor(loss_function=f"Quantile:alpha={alpha:.2f}", **base_params, **hp)
    t0 = time.time()
    reg.fit(xtr, ytr, cat_features=cat_cols)
    fit_sec = time.time() - t0
    pr_tr = np.maximum(reg.predict(xtr), 0.0)
    pr_va = np.maximum(reg.predict(xva), 0.0)
    pr_te = np.maximum(reg.predict(xte), 0.0)

    if shrink_rate > 0:
        k = 1.0 - shrink_rate
        pr_tr, pr_va, pr_te = pr_tr * k, pr_va * k, pr_te * k

    return pr_tr, pr_va, pr_te, fit_sec

## Основной тюнинг



In [6]:
rows = []
for t, p in packs.items():
    for m in mode_defs:
        for hp in param_grid:
            pr_tr, pr_va, pr_te, fit_sec = fit_predict_mode(p, m, hp, shrink_rate=0.0)

            mt_tr = metric_pack(p.y_train, pr_tr)
            mt_va = metric_pack(p.y_val, pr_va)
            mt_te = metric_pack(p.y_test, pr_te)

            r = {'target': t, 'mode': m['mode'], 'fit_sec': fit_sec, **hp}
            for k, v in mt_tr.items():
                r['train_' + k] = v
            for k, v in mt_va.items():
                r['val_' + k] = v
            for k, v in mt_te.items():
                r['test_' + k] = v
            rows.append(r)

res = pd.DataFrame(rows).sort_values(['target', 'val_mae']).reset_index(drop=True)
res.head(20)

,target,mode,fit_sec,depth,learning_rate,l2_leaf_reg,iterations,bagging_temperature,subsample,random_strength,border_count,od_wait,train_mae,train_medae,train_p70_abs_error,train_p90_abs_error,train_r2,train_product_mae,train_engagement_risk_mae,train_small_mae,train_normal_mae,train_long_mae,train_wmape,train_tail_weighted_mae_q90_w2,val_mae,val_medae,val_p70_abs_error,val_p90_abs_error,val_r2,val_product_mae,val_engagement_risk_mae,val_small_mae,val_normal_mae,val_long_mae,val_wmape,val_tail_weighted_mae_q90_w2,test_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,20.517989,7,0.05,8,1100,0.0,0.85,0.5,254,120,189.366918,67.112698,171.626837,495.403107,0.511095,129.656418,134.247255,158.467896,106.515021,732.714062,0.317632,241.219416,232.678288,127.480139,233.495683,572.708731,0.392120,183.438079,186.091711,212.352893,155.871279,818.285876,0.419373,281.079892,272.059931,157.578014,279.131193,657.867322,0.259323,207.629815,212.075539,248.361795,179.218803,865.829381,0.455483,328.700647
1,future_sessions_mean_playtime_7d,train_capped_target,19.619070,7,0.05,3,1100,0.5,1.00,1.0,128,80,190.776107,68.925784,172.026693,493.281865,0.505155,130.869513,135.105260,159.386964,107.245797,739.272069,0.319995,243.181890,232.935713,124.691188,234.784997,574.667077,0.388813,183.333368,187.249141,219.788798,151.492979,821.253347,0.419837,281.416511,268.152447,151.543111,276.046053,647.904737,0.267046,203.256248,209.431378,253.517318,171.366182,861.175453,0.448942,324.665730
2,future_sessions_mean_playtime_7d,train_quantile_04,20.521232,7,0.05,8,800,0.5,0.85,1.0,64,80,208.722122,74.825080,192.458930,530.159281,0.424014,130.132585,134.938524,125.909703,132.161384,870.940651,0.350097,271.659129,233.087270,121.667098,233.391119,571.113670,0.363974,171.924710,176.229989,180.271071,164.606676,889.916535,0.420110,287.646094,269.079069,146.895023,273.083747,640.497123,0.241110,192.368864,197.937706,208.861159,181.094539,929.196347,0.450493,331.828057
3,future_sessions_mean_playtime_7d,train_capped_target,15.515622,7,0.07,8,800,1.0,0.85,2.0,254,120,189.510975,67.836655,172.914388,488.272058,0.514549,130.313735,135.137887,159.886559,107.071931,726.843420,0.317873,240.790620,233.158626,129.108218,232.619359,559.801594,0.395534,181.225991,186.245729,208.655119,159.453111,814.528012,0.420239,281.200556,269.370689,150.713567,276.230546,654.247805,0.262495,201.989236,207.554861,242.578626,175.328116,878.014335,0.450981,327.329734
4,future_sessions_mean_playtime_7d,train_quantile_04,30.675961,7,0.05,3,1100,1.0,0.70,2.0,64,80,198.877385,67.364267,178.455821,510.261047,0.459126,123.542230,128.240682,121.429814,124.063299,835.731119,0.333584,259.443613,233.551458,120.987785,233.765819,578.322543,0.365308,173.911897,178.486788,188.307366,161.902996,881.879089,0.420947,287.357052,269.482272,151.235449,276.670930,641.676905,0.251426,196.837061,201.231781,218.295447,180.536666,911.815722,0.451168,330.507222
5,future_sessions_mean_playtime_7d,train_quantile_04,35.995439,6,0.07,3,1100,1.0,0.85,1.0,64,80,207.968002,72.238425,190.713664,528.444377,0.423524,130.238991,135.014627,127.331825,131.237624,864.689883,0.348832,270.381414,233.858020,120.380938,234.237134,580.022894,0.359064,173.430278,177.436849,185.358736,162.374965,893.733250,0.421499,288.629100,271.827943,150.504733,281.001328,644.011371,0.235127,197.803833,203.096617,219.248005,182.967080,917.269047,0.455095,333.299636
6,future_sessions_mean_playtime_7d,train_quantile_04,29.729966,7,0.05,8,1100,0.5,0.85,0.5,128,80,199.188473,66.361612,179.414401,509.269783,0.452194,124.183438,128.954268,122.669329,124.397069,833.178475,0.334106,259.431753,233.910929,122.884316,237.642360,575.078291,0.372035,172.702117,177.841305,180.993652,167.130004,879.920029,0.421595,287.674624,270.894934,152.474188,273.291192,652.742935,0.241318

Тюним future_sessions_mean_playtime_7d. По нему лучший кандидат для точного прогноза средней длительности сессий за неделю - train_capped_target: минимальный validation MAE 232.68, а на test MAE 272.06 и R2 0.259. Этот режим лучше сохраняет качество на длинном хвосте и дает более стабильную общую ошибку

Для продуктовых CRM-сценариев лучше выглядит train_quantile_04: при небольшом ухудшении общей точности он дает более низкие ProductMAE 192.37 и EngagementRiskMAE 197.94. Он лучше подходит для такиз сценариев, где опаснее переоценить будущую вовлеченность игрока. Более консервативный train_engagement_quantile_035 сильнее снижает ошибку на коротких сессиях, но в этом прогоне уже не дает дополнительного выигрыша относительно quantile 0.40 и заметнее ухудшает MAE и R2

Также видно заметное ухудшение качества: test MAE примерно на 35–40 секунд выше validation MAE. Возможные причины это переобучение и temporal drift. Поэтому дополнительно уменьшим сложность модели и усилим регуляризацию

## Локальный model_shrink_rate around-best

Дополнительный мини-блок вокруг лучших конфигураций, чтобы проверить аккуратное shrink регуляризированиее

In [7]:
best_seed = res.groupby(['target', 'mode'], as_index=False).first()[['target', 'mode', 'depth', 'learning_rate', 'l2_leaf_reg', 'iterations', 'bagging_temperature', 'subsample', 'random_strength', 'border_count', 'od_wait']]
shrink_rates = [0.0, 0.05, 0.1]

rows_shrink = []
for _, br in best_seed.iterrows():
    t = br['target']
    p = packs[t]
    m = [x for x in mode_defs if x['mode'] == br['mode']][0]
    hp = {
        'depth': int(br['depth']),
        'learning_rate': float(br['learning_rate']),
        'l2_leaf_reg': float(br['l2_leaf_reg']),
        'iterations': int(br['iterations']),
        'bagging_temperature': float(br['bagging_temperature']),
        'subsample': float(br['subsample']),
        'random_strength': float(br['random_strength']),
        'border_count': int(br['border_count']),
        'od_wait': int(br['od_wait']),
    }
    for sr in shrink_rates:
        pr_tr, pr_va, pr_te, fit_sec = fit_predict_mode(p, m, hp, shrink_rate=sr)
        mt_te = metric_pack(p.y_test, pr_te)
        rows_shrink.append({
            'target': t,
            'mode': m['mode'],
            'model_shrink_rate': sr,
            'fit_sec': fit_sec,
            'test_mae': mt_te['mae'],
            'test_r2': mt_te['r2'],
            'test_product_mae': mt_te['product_mae'],
            'test_engagement_risk_mae': mt_te['engagement_risk_mae'],
            'test_wmape': mt_te['wmape'],
            'test_tail_weighted_mae_q90_w2': mt_te['tail_weighted_mae_q90_w2'],
        })

shrink_table = pd.DataFrame(rows_shrink).sort_values(['target', 'mode', 'test_mae'])
shrink_table

,target,mode,model_shrink_rate,fit_sec,test_mae,test_r2,test_product_mae,test_engagement_risk_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,0.00,20.401070,272.059931,0.259323,207.629815,212.075539,0.455483,328.700647
1,future_sessions_mean_playtime_7d,train_capped_target,0.05,20.561443,272.059931,0.259323,207.629815,212.075539,0.455483,328.700647
2,future_sessions_mean_playtime_7d,train_capped_target,0.10,19.684128,272.059931,0.259323,207.629815,212.075539,0.455483,328.700647
3,future_sessions_mean_playtime_7d,train_engagement_quantile_035,0.00,13.365929,274.703612,0.219632,193.710481,199.840869,0.459910,338.673967
4,future_sessions_mean_playtime_7d,train_engagement_quantile_035,0.05,13.195404,278.986264,0.200100,193.954010,198.907049,0.467080,345.910775
5,future_sessions_mean_playtime_7d,train_engagement_quantile_035,0.10,12.645224,285.787089,0.174943,196.737686,200.324623,0.478466,355.541341
6,future_sessions_mean_playtime_7d,train_quantile_04,0.00,13.320152,269.079069,0.241110,192.368864,197.937706,0.450493,331.828057
7,future_sessions_mean_playtime_7d,train_quantile_04,0.05,12.993709,272.688680,0.222345,191.756323,196.344868,0.456536,338.363367
8,future_sessions_mean_playtime_7d,train_quantile_04,0.10,12.717286,278.600080,0.197762,192.034974,196.807404,0.466433,347.194188
9,target_next_session_length_sec,train_capped_target,0.00,5.721295,430.440733,0.038543,232.217783,237.009453,0.801172,577.401243


Shrink немного улучшает ProductMAE и EngagementRiskMAE, но ухудшает обычный MAE и R2. Для основного прогноза оставляем shrink 0.00, для осторожного CRM-прогноза можно использовать train_quantile_04 со shrink 0.05. Shrink 0.10 уже слишком сильно ухудшает точность

## Полная таблица запусков

Здесь у каждой строки рядом показываются MAE и R2.

In [8]:
main_cols = [
    'target', 'mode',
    'depth', 'learning_rate', 'l2_leaf_reg', 'iterations',
    'bagging_temperature', 'subsample', 'random_strength', 'border_count', 'od_wait', 'fit_sec',
    'val_mae', 'val_r2', 'test_mae', 'test_r2',
    'val_medae', 'test_medae',
    'val_p70_abs_error', 'test_p70_abs_error',
    'val_p90_abs_error', 'test_p90_abs_error',
    'val_small_mae', 'test_small_mae',
    'val_normal_mae', 'test_normal_mae',
    'val_long_mae', 'test_long_mae',
    'val_product_mae', 'test_product_mae',
    'val_engagement_risk_mae', 'test_engagement_risk_mae',
    'val_wmape', 'test_wmape',
    'val_tail_weighted_mae_q90_w2', 'test_tail_weighted_mae_q90_w2',
]
res[main_cols]

,target,mode,depth,learning_rate,l2_leaf_reg,iterations,bagging_temperature,subsample,random_strength,border_count,od_wait,fit_sec,val_mae,val_r2,test_mae,test_r2,val_medae,test_medae,val_p70_abs_error,test_p70_abs_error,val_p90_abs_error,test_p90_abs_error,val_small_mae,test_small_mae,val_normal_mae,test_normal_mae,val_long_mae,test_long_mae,val_product_mae,test_product_mae,val_engagement_risk_mae,test_engagement_risk_mae,val_wmape,test_wmape,val_tail_weighted_mae_q90_w2,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,7,0.05,8,1100,0.0,0.85,0.5,254,120,20.517989,232.678288,0.392120,272.059931,0.259323,127.480139,157.578014,233.495683,279.131193,572.708731,657.867322,212.352893,248.361795,155.871279,179.218803,818.285876,865.829381,183.438079,207.629815,186.091711,212.075539,0.419373,0.455483,281.079892,328.700647
1,future_sessions_mean_playtime_7d,train_capped_target,7,0.05,3,1100,0.5,1.00,1.0,128,80,19.619070,232.935713,0.388813,268.152447,0.267046,124.691188,151.543111,234.784997,276.046053,574.667077,647.904737,219.788798,253.517318,151.492979,171.366182,821.253347,861.175453,183.333368,203.256248,187.249141,209.431378,0.419837,0.448942,281.416511,324.665730
2,future_sessions_mean_playtime_7d,train_quantile_04,7,0.05,8,800,0.5,0.85,1.0,64,80,20.521232,233.087270,0.363974,269.079069,0.241110,121.667098,146.895023,233.391119,273.083747,571.113670,640.497123,180.271071,208.861159,164.606676,181.094539,889.916535,929.196347,171.924710,192.368864,176.229989,197.937706,0.420110,0.450493,287.646094,331.828057
3,future_sessions_mean_playtime_7d,train_capped_target,7,0.07,8,800,1.0,0.85,2.0,254,120,15.515622,233.158626,0.395534,269.370689,0.262495,129.108218,150.713567,232.619359,276.230546,559.801594,654.247805,208.655119,242.578626,159.453111,175.328116,814.528012,878.014335,181.225991,201.989236,186.245729,207.554861,0.420239,0.450981,281.200556,327.329734
4,future_sessions_mean_playtime_7d,train_quantile_04,7,0.05,3,1100,1.0,0.70,2.0,64,80,30.675961,233.551458,0.365308,269.482272,0.251426,120.987785,151.235449,233.765819,276.670930,578.322543,641.676905,188.307366,218.295447,161.902996,180.536666,881.879089,911.815722,173.911897,196.837061,178.486788,201.231781,0.420947,0.451168,287.357052,330.507222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,target_next_session_length_sec,train_quantile_04,7,0.05,3,800,0.0,0.85,0.5,254,120,21.037168,1119.620794,-2.422390,835.679410,-2.298717,326.938441,236.028032,901.193573,547.554807,3844.463825,3539.502681,1017.467906,673.321379,822.828815,706.858360,2087.090717,1916.647762,393.249320,327.515431,968.374666,692.234132,1.694283,1.555436,1251.640978,951.678979
428,target_next_session_length_sec,train_engagement_quantile_035,6,0.07,8,1100,0.0,1.00,0.5,64,80,25.256184,1471.504981,-5.150616,1496.982329,-8.661207,327.975810,284.339578,992.685013,930.490341,5912.119719,5969.068303,1373.802315,1502.153727,1210.224101,1176.278942,2347.848016,2309.770121,393.744178,400.181934,1333.347312,1419.296526,2.226777,2.786308,1585.575175,1585.992065
429,target_next_session_length_sec,train_engagement_quantile_035,7,0.07,8,500,0.5,0.70,0.5,254,120,14.025079,1473.827500,-4.417228,1491.281974,-7.221537,387.378826,344.421386,1304.796462,1319.580854,5043.525173,5062.048022,1522.748420,1557.021117,1079.607337,1175.538850,2173.865499,2013.225972,446.722194,448.571854,1391.883466,1456.194644,2.230292,2.775698,1582.105188,1553.506872
430,target_next_session_length_sec,train_quantile_04,7,0.07,8,800,1.0,0.85,2.0,254,120,23.121374,1772.749384,-3.706413,1497.967493,-4.789048,2048.921941,1057.941102,2691.709486,2573.780986,2968.494743,2900.858424,1915.657896,1526.882184,1565.723262,1369.282200,1768.912870,1700.956982,661.337121,579.379625,1805.245210,1485.115507,2.682640,2.788142,1788.538236,1524.646532


## Лучшие кандидаты по целям

Формируем витрины по MAE плюс R2, ProductMAE, EngagementRiskMAE и отдельный компромисс.

In [9]:
best_mae = res.sort_values(['test_mae', 'test_r2'], ascending=[True, False]).groupby('target', as_index=False).first()
best_prod = res.sort_values('test_product_mae').groupby('target', as_index=False).first()
best_eng = res.sort_values('test_engagement_risk_mae').groupby('target', as_index=False).first()

best_mae[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae', 'fit_sec']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae,fit_sec
0,future_sessions_mean_playtime_7d,train_capped_target,267.486993,0.275065,203.987703,209.571506,16.022349
1,target_next_session_length_sec,train_capped_target,428.902152,0.048737,232.567823,237.618597,14.291037


In [10]:
best_prod[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae', 'fit_sec']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae,fit_sec
0,future_sessions_mean_playtime_7d,train_engagement_quantile_035,274.453222,0.202604,188.221293,194.876432,21.72916
1,target_next_session_length_sec,train_engagement_quantile_035,439.138351,-0.099040,197.994784,203.483536,10.91971


In [11]:
best_eng[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae', 'fit_sec']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae,fit_sec
0,future_sessions_mean_playtime_7d,train_engagement_quantile_035,274.453222,0.202604,188.221293,194.876432,21.72916
1,target_next_session_length_sec,train_engagement_quantile_035,439.138351,-0.099040,197.994784,203.483536,10.91971


In [12]:
rank_df = res.copy()
rank_cols = ['test_mae', 'test_product_mae', 'test_engagement_risk_mae', 'test_wmape', 'test_tail_weighted_mae_q90_w2']
for c in rank_cols:
    lo, hi = rank_df[c].min(), rank_df[c].max()
    rank_df[c + '_norm'] = (rank_df[c] - lo) / (hi - lo + 1e-12)
rank_df['rank_sum'] = rank_df[[c + '_norm' for c in rank_cols]].sum(axis=1)

best_compromise = rank_df.sort_values('rank_sum').groupby('target', as_index=False).first()
best_compromise[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae', 'rank_sum']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae,rank_sum
0,future_sessions_mean_playtime_7d,train_quantile_04,269.079069,0.241110,192.368864,197.937706,0.022254
1,target_next_session_length_sec,train_quantile_04,430.774565,-0.044367,203.439464,208.685785,0.545374


Единственного победителя нет, выбор зависит от задачи

- Для минимального MAE берем train_capped_target: CRM MAE = 267.49, next-session MAE = 428.90.
- Для product-risk и retention берем train_engagement_quantile_035: у него лучшие ProductMAE и EngagementRiskMAE для обоих таргетов.
- Как компромиссный вариант берем train_quantile_04: он дает балансированное качество

У next-session моделей R2 около нуля или отрицательный. Значит, прогноз следующей конкретной сессии остается шумным, поэтому для CRM-решений надежнее опираться на недельный таргет.

## CRM таргеты для retention слоя


In [5]:
import numpy as np
import pandas as pd

df = pd.read_parquet(
    "/Users/avals282006/Downloads/sessions_preprocessed.parquet"
)

In [6]:
def add_crm_retention_targets(df, installation_col='installation_id', time_col='start', duration_col='duration_seconds', horizon_days=7):
    x = df.copy()
    x[time_col] = pd.to_datetime(x[time_col], errors='coerce')
    x = x[x[time_col].notna() & x[installation_col].notna()].copy()
    x = x.sort_values([installation_col, time_col]).reset_index(drop=True)

    sec_day = 24 * 3600
    delta_ns = np.int64(horizon_days * sec_day * 1_000_000_000)
    dur = pd.to_numeric(x[duration_col], errors='coerce').fillna(0.0).to_numpy(dtype=np.float64)

    fut_play = np.zeros(len(x), dtype=np.float64)
    fut_cnt = np.zeros(len(x), dtype=np.int32)

    for _, idx in x.groupby(installation_col, sort=False).indices.items():
        idx = np.asarray(idx, dtype=np.int64)
        t = x.iloc[idx][time_col].values.astype('datetime64[ns]').astype(np.int64)
        d = dur[idx]
        n = len(idx)

        nxt = np.arange(n, dtype=np.int64) + 1
        r = np.searchsorted(t, t + delta_ns, side='right')
        c = np.maximum(r - nxt, 0)
        fut_cnt[idx] = c

        csum = np.concatenate(([0.0], np.cumsum(d)))
        s = csum[r] - csum[np.minimum(nxt, n)]
        fut_play[idx] = s

    max_t_ns = x[time_col].max().value
    cut_ns = max_t_ns - delta_ns
    observed = x[time_col].values.astype('datetime64[ns]').astype(np.int64) <= cut_ns

    x['future_sessions_count_7d'] = np.where(observed, fut_cnt, np.nan)
    x['future_playtime_7d'] = np.where(observed, fut_play, np.nan)
    x['future_sessions_mean_playtime_7d_v2'] = np.where(observed, np.divide(fut_play, np.maximum(fut_cnt, 1)), np.nan)
    x['churn_7d'] = np.where(observed, (fut_cnt == 0).astype(int), np.nan)
    return x


## Применение и проверки

Считаем таргеты и проверяем диапазоны, долю пропусков и базовые статистики.


In [7]:
crm_df = add_crm_retention_targets(df)

check = pd.DataFrame({
    'target': ['churn_7d', 'future_sessions_mean_playtime_7d_v2'],
    'na_rate': [crm_df['churn_7d'].isna().mean(), crm_df['future_sessions_mean_playtime_7d_v2'].isna().mean()],
    'min': [crm_df['churn_7d'].min(skipna=True), crm_df['future_sessions_mean_playtime_7d_v2'].min(skipna=True)],
    'mean': [crm_df['churn_7d'].mean(skipna=True), crm_df['future_sessions_mean_playtime_7d_v2'].mean(skipna=True)],
    'median': [crm_df['churn_7d'].median(skipna=True), crm_df['future_sessions_mean_playtime_7d_v2'].median(skipna=True)],
    'max': [crm_df['churn_7d'].max(skipna=True), crm_df['future_sessions_mean_playtime_7d_v2'].max(skipna=True)],
})
check


,target,na_rate,min,mean,median,max
0,churn_7d,0.020531,0.0,0.087783,0.0,1.0
1,future_sessions_mean_playtime_7d_v2,0.020531,0.0,587.948784,462.0,7858.0


## Отдельное обучение бустинга для churn 7d

Регрессионные стратегии выше оставляем без изменений: quantile 04, engagement 035 и capped target

Для churn 7d добавляем отдельную классификационную витрину, потому что это бинарный риск удержания

Мы сравниваем три режима с теми же именами стратегий

In [8]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)

def prep_for_churn(df_in, feature_cols, max_rows=30000):
    x = df_in.copy()
    x["start"] = pd.to_datetime(x["start"], errors="coerce")
    x = x[x["start"].notna() & x["churn_7d"].notna()].copy()
    x = x.sort_values("start").reset_index(drop=True)
    if len(x) > max_rows:
        x = x.tail(max_rows).reset_index(drop=True)

    n = len(x)
    i1 = int(n * 0.70)
    i2 = int(n * 0.85)

    tr = x.iloc[:i1].copy()
    va = x.iloc[i1:i2].copy()
    te = x.iloc[i2:].copy()

    use_cols = [c for c in feature_cols if c in x.columns]
    xtr = tr[use_cols].copy()
    xva = va[use_cols].copy()
    xte = te[use_cols].copy()

    ytr = tr["churn_7d"].astype(int).values
    yva = va["churn_7d"].astype(int).values
    yte = te["churn_7d"].astype(int).values

    num_cols = xtr.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = [c for c in xtr.columns if c not in num_cols]

    if num_cols:
        med = xtr[num_cols].median()
        xtr[num_cols] = xtr[num_cols].fillna(med)
        xva[num_cols] = xva[num_cols].fillna(med)
        xte[num_cols] = xte[num_cols].fillna(med)

    if cat_cols:
        xtr[cat_cols] = xtr[cat_cols].astype("object").fillna("unknown")
        xva[cat_cols] = xva[cat_cols].astype("object").fillna("unknown")
        xte[cat_cols] = xte[cat_cols].astype("object").fillna("unknown")

    return xtr, xva, xte, ytr, yva, yte, cat_cols

def cls_pack(y_true, prob, thr=0.5):
    p = np.clip(np.asarray(prob, dtype=float), 1e-6, 1-1e-6)
    y = np.asarray(y_true, dtype=int)
    pred = (p >= thr).astype(int)

    out = {
        "roc_auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "logloss": float(log_loss(y, p, labels=[0,1])),
        "brier": float(brier_score_loss(y, p)),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "pred_pos_rate": float(pred.mean()),
        "true_pos_rate": float(y.mean()),
    }
    return out


In [10]:
import time
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)


drop_cols = {'appmetrica_device_id', 'installation_id', 'session_id', 'start', 'end', 'session_date', 'install_datetime', 'prev_session_end', 'duration_hms', 'churn_7d'}
churn_feature_cols = [c for c in crm_df.columns if c not in drop_cols and not c.lower().startswith('target') and not c.lower().startswith('future_')]
xtr, xva, xte, ytr, yva, yte, cat_cols_ch = prep_for_churn(crm_df, churn_feature_cols)

hp_ch = {
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5,
    "iterations": 500,
    "bagging_temperature": 0.5,
    "subsample": 0.85,
    "random_strength": 1.0,
    "border_count": 128,
    "od_type": "Iter",
    "od_wait": 80,
}

churn_modes = [
    {"mode": "train_capped_target", "class_weights": [1.0, 1.0]},
    {"mode": "train_quantile_04", "class_weights": [1.0, 1.5]},
    {"mode": "train_engagement_quantile_035", "class_weights": [1.0, 2.0]},
]

rows_ch = []
for m in churn_modes:
    t0 = time.time()
    clf = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=m["class_weights"],
        random_seed=42,
        verbose=False,
        thread_count=-1,
        **hp_ch,
    )
    clf.fit(xtr, ytr, cat_features=cat_cols_ch)
    fit_sec = time.time() - t0

    pva = clf.predict_proba(xva)[:, 1]
    pte = clf.predict_proba(xte)[:, 1]

    va_m = cls_pack(yva, pva, thr=0.5)
    te_m = cls_pack(yte, pte, thr=0.5)

    r = {"mode": m["mode"], "fit_sec": fit_sec, "thr": 0.5}
    for k, v in va_m.items():
        r["val_" + k] = v
    for k, v in te_m.items():
        r["test_" + k] = v
    rows_ch.append(r)

churn_res = pd.DataFrame(rows_ch).sort_values(["test_roc_auc", "test_pr_auc"], ascending=[False, False]).reset_index(drop=True)
churn_res


,mode,fit_sec,thr,val_roc_auc,val_pr_auc,val_logloss,val_brier,val_accuracy,val_balanced_accuracy,val_precision,...,test_pr_auc,test_logloss,test_brier,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1,test_pred_pos_rate,test_true_pos_rate
0,train_quantile_04,15.331111,0.5,0.809418,0.289666,0.239453,0.068390,0.910444,0.593197,0.413613,...,0.346297,0.238931,0.067877,0.913111,0.633093,0.480851,0.295812,0.366288,0.052222,0.084889
1,train_capped_target,16.046241,0.5,0.810183,0.290278,0.233716,0.065909,0.916222,0.526219,0.431373,...,0.347194,0.230941,0.064748,0.916444,0.552978,0.536585,0.115183,0.189655,0.018222,0.084889
2,train_engagement_quantile_035,13.698742,0.5,0.809510,0.286842,0.243185,0.069721,0.906667,0.588679,0.377451,...,0.340737,0.246130,0.070222,0.908889,0.630786,0.444882,0.295812,0.355346,0.056444,0.084889


## Итог по churn 7d

Лидера по churn выбираем по ROC AUC и PR AUC, а стабильность проверяем по logloss и brier.
Это отдельная витрина для retention риска, которую затем можно склеить с регрессионным CRM таргетом.


In [11]:
churn_best = churn_res.sort_values(["test_roc_auc", "test_pr_auc"], ascending=[False, False]).head(1)
churn_best[[
    "mode", "test_roc_auc", "test_pr_auc", "test_logloss", "test_brier",
    "test_accuracy", "test_balanced_accuracy", "test_precision", "test_recall", "test_f1",
    "test_pred_pos_rate", "test_true_pos_rate", "fit_sec"
]]


,mode,test_roc_auc,test_pr_auc,test_logloss,test_brier,test_accuracy,test_balanced_accuracy,test_precision,test_recall,test_f1,test_pred_pos_rate,test_true_pos_rate,fit_sec
0,train_quantile_04,0.82035,0.346297,0.238931,0.067877,0.913111,0.633093,0.480851,0.295812,0.366288,0.052222,0.084889,15.331111


## Мини пример CRM выхода

Для уверенности показываем минимальный пример двух ключевых полей CRM слоя.


In [12]:
ex = crm_df[['installation_id', 'churn_7d', 'future_sessions_mean_playtime_7d_v2']].dropna().head(1).copy()
r = ex.iloc[0]
sample = {
    'player_id': str(r['installation_id']),
    'churn_probability_7d': float(r['churn_7d']),
    'expected_mean_session_length_7d_sec': float(r['future_sessions_mean_playtime_7d_v2']),
}
sample


{'player_id': '0000366d3dd84eb19885b217d065f6f7',
 'churn_probability_7d': 1.0,
 'expected_mean_session_length_7d_sec': 0.0}

модель train_quantile_04 нормально ранжирует риск ухода (ROC AUC = 0.82). PR AUC = 0.35 выглядит не оч, но это ожидаемо: реальный churn есть только у 8.5% игроков.

При пороге 0.5 модель достаточно осторожная:
- находит около 29.6% ушедших игроков
- из отмеченных как риск около 48.1% действительно уходят
- помечает как риск только 5.2% игроков.


## Общий вывод

В ноутбуке сравнили три стратегии CatBoost для прогноза следующей сессии и средней длительности будущих сессий за 7 дней.

- Для минимальной ошибки в секундах лучше использовать train_capped_target. 
- Для short-risk и retention-сценариев лучше подходит train_engagement_quantile_035: он осторожнее оценивает вовлеченность и дает лучшие ProductMAE и EngagementRiskMAE. 
- Компромиссный вариант — train_quantile_04.

Shrink почти не улучшает результаты: для основной модели оставляем shrink 0.00. Дополнительно обучили churn_7d модель. Лучший режим train_quantile_04 показал ROC AUC = 0.82 и используется для оценки вероятности ухода игрока.

## Decision rules

- Для точного прогноза секунд — train_capped_target.
- Для осторожных short-risk решений — train_engagement_quantile_035.
- Для сбалансированного CRM-прогноза — train_quantile_04.
- Для оценки риска ухода — отдельная churn_7d модель train_quantile_04.
- Для CRM используем вместе churn_probability_7d и expected_mean_session_length_7d_sec.

## Вывод по гиперпараметрам и стратегиям

Лучшие конфигурации чаще используют depth = 7, learning_rate = 0.05–0.07 и 800–1100 итераций. 

Дальнейшее увеличение сложности не выглядит полезным: разрыв между train, validation и test уже заметен, поэтому расширение глубины и числа деревьев может усилить переобучение.

Shrink не улучшает основную модель, поэтому оставляем shrink = 0.00.